<a href="https://colab.research.google.com/github/hj245668/ds6_warpUp/blob/main/warpUp1_note.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# note
# warp up 1회차
# KRX & S&P500 데이터 분석
# pandas 기본 문법 / 주가 데이터 기본 및 상관관계/ 조합 공식/ 이항분포 응용
# 날짜: 2025-10-21

# --- 라이브러리 불러오기
import FinanceDataReader as fdr
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import binom
import itertools

# --- 데이터 불러오기
# 한국 KRX와 미국 S&P500 지수 데이터
krx = fdr.StockListing('KRX')
spx = fdr.StockListing('S&P500')

# 예시: 삼성전자와 S&P500 지수
start, end = '2022-01-01', '2025-10-21'
df_krx = fdr.DataReader('005930', start, end)
df_spx = fdr.DataReader('US500', start, end)

# --- 상관관계 분석 (Correlation Analysis)
# 날짜를 숫자형으로 변환
df_krx['date_num'] = pd.to_datetime(df_krx.index).map(pd.Timestamp.toordinal)

# 상관계수 계산
corr = df_krx.corrwith(df_krx['date_num']).sort_values(ascending=False)

# DataFrame 변환
corr_df = corr.reset_index()
corr_df.columns = ['Feature', 'Correlation']
corr_df = corr_df.dropna().sort_values('Correlation', ascending=True)

# --- 시각화
plt.figure(figsize=(8, 5))
sns.barplot(data=corr_df, x='Correlation', y='Feature', palette='coolwarm')
plt.title('Correlation of Each Feature with Date', fontsize=14)
plt.xlabel('Correlation Coefficient')
plt.ylabel('Feature')
plt.grid(True, axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# --- 조합 공식 (Combination Formula)
from math import factorial

def combination(n, k):
    return factorial(n) / (factorial(k) * factorial(n - k))

# 예시: 5일 중 3일 선택
n, k = 5, 3
comb = combination(n, k)
print(f"5일 중 3일을 선택하는 조합 수: {comb:.0f}")

# --- 이항분포 (Binomial Distribution)
p = 0.5  # 하루 상승 확률
prob = binom.pmf(k, n, p)
print(f"5일 중 3일 상승할 확률: {prob:.4f}")

# -------------------------------------------------------------
# --- 확장 rollCor.ipynb: 이동상관계수 (Rolling Correlation)
df_merged = pd.DataFrame({
    'KRX_Close': df_krx['Close'],
    'SPX_Close': df_spx['Close']
}).dropna()

rolling_corr = df_merged['KRX_Close'].rolling(30).corr(df_merged['SPX_Close'])

plt.figure(figsize=(10, 5))
rolling_corr.plot(color='purple')
plt.title('30일 이동 상관관계 (KRX vs S&P500)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Correlation')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# --- 출력
print("\n 분석 요약")
print("- 상관관계 높은 지표: high/open/market cap → 장기 상승 추세")
print("- 낮은 상관관계 지표: trading volume, fluctuation rate → 단기 변동성 중심")
print("- 조합공식과 이항분포를 통해 상승 패턴의 확률적 구조 해석 가능")
